In [1]:
# Импортируем библиотеку pandas для работы с табличными данными
import pandas as pd

# Загружаем исходный датасет Telco Customer Churn по прямой ссылке
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

In [2]:
# Фильтруем столбцы датафрейма, выбирая только те, которые содержат текстовый тип данных ('object')
obj_cols = df.select_dtypes(include='object').columns.tolist()

# Выводим общее количество найденных текстовых колонок
print('Total object columns:', len(obj_cols))

# Отображаем получившийся список названий категориальных признаков
print(obj_cols)

Total object columns: 18
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


In [3]:
# Смотрим распределение уникальных значений в колонке 'gender'
df['gender'].value_counts()

,count
gender,
Male,3555
Female,3488


In [4]:
# Применяем One-Hot Encoding к признаку 'InternetService' с добавлением префикса
dummies = pd.get_dummies(df['InternetService'], prefix='Internet')

# Выводим размерность получившейся разреженной матрицы (количество строк и новых колонок)
print('Shape:', dummies.shape)

# Отображаем первые 5 строк полученного DataFrame
dummies.head()

Shape: (7043, 3)


,Internet_DSL,Internet_Fiber optic,Internet_No
0,True,False,False
1,True,False,False
2,True,False,False
3,True,False,False
4,False,True,False


In [5]:
# Делаем то же самое кодирование, но удаляем первый столбец для предотвращения мультиколлинеарности
dummies2 = pd.get_dummies(df['InternetService'], prefix='Internet', drop_first=True)

# Проверяем новую размерность, чтобы убедиться, что одна колонка была успешно удалена
print('With drop_first:', dummies2.shape)

# Показываем первые строки обновленного датафрейма
dummies2.head()

With drop_first: (7043, 2)


,Internet_Fiber optic,Internet_No
0,False,False
1,False,False
2,False,False
3,False,False
4,True,False


In [6]:
# Импортируем OneHotEncoder из библиотеки scikit-learn для кодирования категориальных признаков
from sklearn.preprocessing import OneHotEncoder

# Инициализируем энкодер, указав sparse_output=False для получения обычного массива вместо разреженной матрицы
ohe = OneHotEncoder(sparse_output=False)

# Обучаем энкодер и преобразуем текстовую колонку 'TechSupport' в бинарные столбцы
encoded = ohe.fit_transform(df[['TechSupport']])

# Создаем новый DataFrame из закодированных данных, автоматически извлекая новые имена для колонок
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['TechSupport']))

# Выводим первые строки полученной таблицы для проверки результата
encoded_df.head()

,TechSupport_No,TechSupport_No internet service,TechSupport_Yes
0,1.0,0.0,0.0
1,1.0,0.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0


In [7]:
# Создаем бинарный признак 'Partner_bin', заменяя текстовые значения 'Yes' и 'No' на числа 1 и 0 с помощью словаря
df['Partner_bin'] = df['Partner'].map({'No': 0, 'Yes': 1})

# Отображаем первые строки исходной и новой колонок рядом, чтобы убедиться в корректности маппинга
df[['Partner', 'Partner_bin']].head()


,Partner,Partner_bin
0,Yes,1
1,No,0
2,No,0
3,No,0
4,No,0


In [8]:
# Применяем One-Hot Encoding сразу к нескольким категориальным признакам с удалением первого столбца для каждого
multi = pd.get_dummies(df[['gender', 'InternetService', 'TechSupport', 'Contract']], drop_first=True)

# Выводим финальную размерность получившейся таблицы (ожидается около 7-8 колонок)
print('Shape:', multi.shape)

# Отображаем первые 5 строк преобразованного датафрейма
multi.head()

Shape: (7043, 7)


,gender_Male,InternetService_Fiber optic,InternetService_No,TechSupport_No internet service,TechSupport_Yes,Contract_One year,Contract_Two year
0,False,False,False,False,False,False,False
1,True,False,False,False,False,True,False
2,True,False,False,False,False,False,False
3,True,False,False,False,True,True,False
4,False,True,False,False,False,False,False
